# A2UI Jupyter manual smoke test

Run the form cell, enter a name, and click **Submit**. The callback replaces the form with a confirmation after the cell finishes. Run the next cell to inspect the action.

If a saved output reports that its widget model was not found, clear the output and rerun the form cell. See the [README](../README.md#saved-widget-output) for saving widget state.

In [ ]:
from IPython.display import display

from a2ui_jupyter import A2UISurface

surface = A2UISurface()
surface.send(
    [
        {
            "version": "1.0",
            "createSurface": {
                "surfaceId": "manual-smoke-test",
                "catalogId": "https://a2ui.org/specification/v1_0/catalogs/basic/catalog.json",
            },
        },
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "manual-smoke-test",
                "path": "/person/name",
                "value": "",
            },
        },
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "manual-smoke-test",
                "path": "/status",
                "value": "Waiting for submission",
            },
        },
        {
            "version": "1.0",
            "updateComponents": {
                "surfaceId": "manual-smoke-test",
                "components": [
                    {
                        "id": "root",
                        "component": "Column",
                        "children": ["title", "name", "submit", "status"],
                        "align": "stretch",
                    },
                    {
                        "id": "title",
                        "component": "Text",
                        "text": "A2UI JupyterLab smoke test",
                    },
                    {
                        "id": "name",
                        "component": "TextField",
                        "label": "Name",
                        "placeholder": "Enter a name",
                        "value": {"@path": "/person/name"},
                    },
                    {
                        "id": "submit",
                        "component": "Button",
                        "child": "submit_label",
                        "variant": "primary",
                        "action": {
                            "event": {
                                "name": "submit",
                                "context": {
                                    "name": {"@path": "/person/name"},
                                },
                            },
                        },
                    },
                    {
                        "id": "submit_label",
                        "component": "Text",
                        "text": "Submit",
                    },
                    {
                        "id": "status",
                        "component": "Text",
                        "text": {"@path": "/status"},
                    },
                ],
            },
        },
    ]
)


action_data = None


def handle_action(action):
    global action_data
    action_data = action.model_dump(by_alias=True, exclude_unset=True)
    # Replace the form with a confirmation so it cannot be submitted again.
    surface.send(
        [
            {
                "version": "1.0",
                "updateDataModel": {
                    "surfaceId": "manual-smoke-test",
                    "path": "/status",
                    "value": f"Submitted: {action_data['action']['context']['name']}",
                },
            },
            {
                "version": "1.0",
                "updateComponents": {
                    "surfaceId": "manual-smoke-test",
                    "components": [
                        {
                            "id": "root",
                            "component": "Column",
                            "children": ["title", "status"],
                            "align": "stretch",
                        },
                    ],
                },
            },
        ]
    )


surface.on_action(handle_action)
display(surface)

After submitting, inspect the action:

In [ ]:
action_data